<a href="https://colab.research.google.com/github/Yusef-Hazem/Lang-Frameworks/blob/main/02QueryTransformations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Python 3.12.13
!pip install -U langchain==1.3.11
!pip install requests==2.34.2
!pip install -U langchain_community==0.4.2
!pip install -U python-dotenv==1.2.2
!pip install -U langchain-huggingface==1.2.2
!pip install faiss-cpu==1.14.3
!pip install langchain-groq==1.1.3

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import bs4
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

True

In [ ]:
# set enviroment variable of the groq APi key
# gsk_DkOInQs0mmiT7z2dlNNBWGdyb3FY021sV6oCN80L55hpKbdX7Q9vxx
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
loader = WebBaseLoader(web_path="https://lilianweng.github.io/posts/2023-06-23-agent/" ,
                       bs_kwargs={
                           "parse_only"  :bs4.SoupStrainer( class_ = ("post-single" , "post-content" , "post-title"))
                        })
doc = loader.load()

In [ ]:
# chunking

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(doc)


In [ ]:
#embedding
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}

embed_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def format_retrived_docs (docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
#vector DB
vector_db = FAISS.from_documents(documents = chunks , embedding = embed_model)
retriver = vector_db.as_retriever() | format_retrived_docs

In [ ]:
template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions each in one line . Original question: {question}"""



prompt = ChatPromptTemplate.from_template(template)

In [ ]:
llm= ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
generate_5_versions_query = (
    prompt |
    llm |
    StrOutputParser() |
    (lambda x : x.split("\n"))
)


MultiQuery Generation

In [ ]:
def get_unique_union(documents):
    seen = set()
    unique_docs = []

    for sublist in documents:
        for doc in sublist:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique_docs.append(doc)

    return unique_docs

In [ ]:
# get all the docs related to the seviral questions and union it , finally delete redundant ones.
question = "what is the meaning of Decomposition ?"

retrival_chain = generate_5_versions_query | retriver.map() | get_unique_union

In [ ]:
docs = retrival_chain.invoke({"question" : question})

In [ ]:
# here we can bulid our rag chain
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm_augmantaion = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
final_rag_chain = (
    {"context": retrival_chain , "question": RunnablePassthrough()} | prompt | llm_augmantaion |StrOutputParser()
)


In [ ]:
print(final_rag_chain.invoke(question))

Based on the provided context, decomposition refers to the process of breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks. This is also referred to as "task decomposition". It can be done in various ways, such as using large language models (LLMs) with simple prompting, using task-specific instructions, or with human inputs.


RAG Fusion

In [ ]:

def rank_retrived_docs(documents_list= list[list]  , k =60) :
  fused_scores = dict()

  for docs in documents_list :
    for rank , doc in enumerate(docs) :
      key =(doc.page_content ,
            frozenset(doc.metadata.items()))

      if key not in fused_scores:
        fused_scores[key] = {
            "document": doc,
            "score": 0.0,
        }

      fused_scores[key]["score"] += 1 / (rank + k)


  ranked_resualts = sorted(
      ( (item['document'] , item['score'])
      for item in fused_scores.values()) ,
      key= lambda x : x[1] ,
      reverse=True)
  return ranked_resualts

In [ ]:
retrive_ranked_docs_chain = (
    generate_5_versions_query | retriver.map() |rank_retrived_docs
)

In [ ]:
ranked_docs =retrive_ranked_docs_chain.invoke(question)

In [ ]:
len(ranked_docs)

10

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm_augmantaion = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
final_rag_fusion_chain = (
    {"context": retrive_ranked_docs_chain, "question": RunnablePassthrough()} | prompt | llm_augmantaion |StrOutputParser()
)


In [ ]:
final_rag_fusion_chain.invoke("what is the writer of this document ?")

"The writer of this document is Lilian Weng, as indicated by the metadata 'source' which points to 'https://lilianweng.github.io/posts/2023-06-23-agent/'."

Decomposition

In [ ]:
template  = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Only Output must be (3 queries):"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
get_queries = prompt | llm |StrOutputParser()| (lambda x : x.split("\n"))
question = "What are the main components of an LLM-powered autonomous agent system?"
questions = get_queries.invoke({"question" : question})

In [ ]:
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question:

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
and make it very summarized
"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
def format_qa_pair(question, answer):
    """Format Q and A pair"""

    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

In [ ]:
from operator import itemgetter

q_a_pairs = ""
for q in questions :
  final_decomposition_chain = (
    {"context" : itemgetter("question") |retriver ,
    "question" : itemgetter("question") ,
    "q_a_pairs" :itemgetter("q_a_pairs") } |
    prompt |
    llm|
    StrOutputParser()
  )

  answer = final_decomposition_chain.invoke({"question" : q , "q_a_pairs" : q_a_pairs})
  q_a_pair = format_qa_pair(q , answer)
  q_a_pairs += (   q_a_pair + "\n---\n")



In [ ]:
answer

'The provided context does not explicitly mention specific software frameworks and architectures used for LLM-powered autonomous agent systems. However, it discusses components such as planning, memory, and tool use, which may utilize various AI frameworks and architectures, but specific details are not provided.'

Answer Individually

In [ ]:
questions

['1. What are the key hardware components required for an LLM-powered autonomous agent system?',
 '2. How do LLMs integrate with other AI components, such as computer vision and sensor processing, in an autonomous agent system?',
 '3. What software frameworks and architectures are commonly used to develop and deploy LLM-powered autonomous agent systems?']

In [ ]:

def get_anwers_of_list_questions(questions: list , retriver : object ) :
  rag_resualts =[]
  llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
  template = """ You are ai assistant , please answer this question : {question}\n\n based on the context : {context} """
  prompt = ChatPromptTemplate.from_template(template)
  for q in questions :
    answer = (
        {"question" : itemgetter("question") , "context" : itemgetter("question") | retriver } |
        prompt | llm | StrOutputParser()
    ).invoke({"question" : q})

    rag_resualts.append(answer)


  return rag_resualts



In [ ]:
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
def format_qa_pairs (questions : list , answers :list) :
  formated_string = ""
  for i,(q , a) in enumerate(zip(questions, answers) , start=1) :
    formated_string += f"Question {i}: {q}\nAnswer {i}: {a}\n\n"

  return formated_string.strip()


In [ ]:
question

'What are the main components of an LLM-powered autonomous agent system?'

In [ ]:
questions

['1. What are the key hardware components required for an LLM-powered autonomous agent system?',
 '2. How do LLMs integrate with other AI components, such as computer vision and sensor processing, in an autonomous agent system?',
 '3. What software frameworks and architectures are commonly used to develop and deploy LLM-powered autonomous agent systems?']

In [ ]:
questions_answers  = get_anwers_of_list_questions(questions , retriver)
context = format_qa_pairs(questions , questions_answers)
final_answer_indvidual_chain =(
    {"context" : itemgetter("context") , "question" : itemgetter("question")} |
    prompt | llm | StrOutputParser()
)

final_answer_indvidual_chain.invoke({"context" : context , "question" : question})

'Based on the provided Q+A pairs, the main components of an LLM-powered autonomous agent system can be synthesized as follows:\n\n1. **Hardware Components**: The system requires high-performance computing infrastructure, large memory and storage, networking and communication infrastructure, specialized hardware for specific tasks, robust power supply and cooling systems, and secure and reliable data storage.\n2. **LLM Integration**: The LLM integrates with other AI components, such as computer vision and sensor processing, through multimodal fusion, sensor data processing, API calls, hybrid architecture, and modular design.\n3. **Software Frameworks and Architectures**: The system utilizes software frameworks and architectures like Transformers, PyTorch, TensorFlow, RLlib, Stable Baselines, microservices architecture, API-based frameworks, graph-based architectures, and modular architecture to develop and deploy the LLM-powered autonomous agent system.\n\nThe main components of an LLM-

Step Back

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]


example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}") ,
    ("ai", "{output}") ])
fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
fewshot_prompt

FewShotChatMessagePromptTemplate(examples=[{'input': 'Could the members of The Police perform lawful arrests?', 'output': 'what can the members of The Police do?'}, {'input': 'Jan Sindel’s was born in what country?', 'output': 'what is Jan Sindel’s personal history?'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['output'], input_types={}, partial_variables={}, template='{output}'), additional_kwargs={})]))

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        fewshot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)


generate_step_back_query = (
    prompt | llm | StrOutputParser()
)

generate_step_back_query.invoke({"question" : question})

'What are the key parts of an artificial intelligence system?'

In [ ]:
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)


In [ ]:
question

'What are the main components of an LLM-powered autonomous agent system?'

In [ ]:
final_step_back_chain = (

{"normal_context" : itemgetter("question") | retriver ,
 "step_back_context" : itemgetter("question") | generate_step_back_query | retriver ,
 "question"  : itemgetter("question") } |
response_prompt | llm | StrOutputParser()
)
final_step_back_chain.invoke({"question" : question})

'The main components of an LLM-powered autonomous agent system are Planning, Memory, and Tool Use.\n\nTo break it down, the process of understanding and analyzing the components of an LLM-powered autonomous agent system involves several stages. \n\nGiven the user\'s input: "What are the main components of an LLM-powered autonomous agent system?", the task planning stage identifies the key components that make up such a system. \n\nIn the model selection stage, I recognize that the relevant information can be found in the context provided, specifically in the "Agent System Overview" section, which outlines the three primary components.\n\nDuring the task execution stage, I analyze the context and identify the three main components: \n1. **Planning**: This component involves task decomposition and self-reflection, enabling the agent to break down complex tasks into manageable parts and evaluate its own performance.\n2. **Memory**: This component encompasses various types of memory and ut

HyDE  - > hypothetical document


In [ ]:
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""



prompt= ChatPromptTemplate.from_template(template)

genrate_hyde_query = (
    prompt | llm | StrOutputParser()
)

genrate_hyde_query.invoke({"question" : question})

"The development of Large Language Model (LLM)-powered autonomous agent systems has revolutionized the field of artificial intelligence, enabling the creation of sophisticated, self-governing entities capable of interacting with and adapting to complex environments. At its core, an LLM-powered autonomous agent system consists of several key components that work in tandem to facilitate autonomous decision-making and action execution.\n\nFirstly, the **Large Language Model (LLM) module** serves as the cognitive backbone of the system, providing the agent with the ability to process, understand, and generate human-like language. This module is typically based on a transformer-based architecture, such as BERT or RoBERTa, which has been trained on vast amounts of text data to acquire a deep understanding of linguistic structures and relationships. The LLM module enables the agent to comprehend and respond to natural language inputs, as well as generate coherent and contextually relevant tex

In [ ]:
doc_to_retrive = genrate_hyde_query.invoke({"question" : question})

retrived_docs = retriver.invoke( doc_to_retrive)
retrived_docs

'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nOr\n@article{weng2023agent,\n  title   = "LLM-powered Autonomous Agents",\n  author  = "Weng, Lilian",\n  journal = "lilianweng.github.io",\n  year    = "2023",\n  month   = "Jun",\n  url     = "https://lilianweng.github.io/posts/2023-06-23-agent/"\n}\nReferences#\n[1] Wei et al. “Chain of thought prompting elicits reasoning in large language models.” NeurIPS 2022\n[2] Yao et al. “Tree of Thoughts: Dliberate Problem Solving with Large Language Models.” arXiv preprint arXiv:2305.10601 (2023).\n\n[3

In [ ]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_hyde_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_hyde_chain.invoke({"context":retrived_docs,"question":question})

'The main components of an LLM-powered autonomous agent system are not explicitly listed in the provided context. However, it is mentioned that the LLM functions as the agent\'s brain, complemented by "several key components", but these components are not specified. To provide a complete answer, more information from the original text or additional context would be necessary.'